In [ ]:
import os
import types
import datetime
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import sys;
sys.path.append(os.getcwd())
sys.path.append(os.path.join(os.getcwd(), 'datasets'))
from datasets import ruijinreader

In [ ]:
sleep_datasets = {
    'ruijin' : {
        'data_dir' : 'g:/PSG/',
        'output_dir' : 'g:/PSG/output/',
    },
    # 'ruijin' : {
    #     'data_dir' : '/home/yuty2009/data/eegdata/sleep/ruijin/',
    #     'output_dir' : '/home/yuty2009/data/eegdata/sleep/ruijin/output/',
    # },
}

In [ ]:
args = types.SimpleNamespace()

args.dataset = 'ruijin'
args.feature = 'features_yasa'
args.data_dir = sleep_datasets[args.dataset]['data_dir']
args.output_dir = sleep_datasets[args.dataset]['output_dir']

args.arch = 'kmeans'
output_prefix = f"{args.arch}_psg"
output_prefix += "/session_" + datetime.datetime.now().strftime("%Y%m%d%H%M%S")
if not hasattr(args, 'output_dir'):
    args.output_dir = args.data_dir
args.output_dir = os.path.join(args.output_dir, output_prefix)
os.makedirs(args.output_dir)

# Data loading code
print("=> loading dataset {} from '{}'".format(args.dataset, args.data_dir))
args.target_labels = None
if args.dataset == 'ruijin':
    args.num_classes = 2
    # df_labels = pd.read_csv(os.path.join(args.data_dir, 'subject_labels_morethan2h.csv'))
    df_labels = pd.read_csv(os.path.join(args.data_dir, 'subject_labels_sleep.csv'))
    data, labels_stage, subjects = ruijinreader.load_dataset_preprocessed(
        args.data_dir+f"{args.feature}/",
        subjects=df_labels['subid'].values
    )
    labels = [df_labels.loc[df_labels['subid'] == subid, 'label'].values[0] for subid in subjects]
    df_labels_hy = pd.read_csv(os.path.join(args.data_dir, 'subject_labels_with_hy_morethan2h.csv'))
    subjects_hy = df_labels_hy['subid'].values
    labels_hy = [df_labels_hy.loc[df_labels_hy['subid'] == subid, 'label_hy'].values[0] for subid in subjects_hy]
    labels_hy = np.array(labels_hy, dtype=int)
    df_labels_hy_new = pd.read_csv(os.path.join(args.data_dir, 'subject_labels_with_hy_msa_pd_new_morethan2h.csv'))
    subjects_hy_new = df_labels_hy_new['subid'].values
    labels_hy_new = [df_labels_hy_new.loc[df_labels_hy_new['subid'] == subid, 'label_hy'].values[0] for subid in subjects_hy_new]
    labels_hy_new = np.array(labels_hy_new, dtype=int)
    args.target_labels = [2, 1] # 0: HC, 1: MSA, 2: PD, 3: RBD
else:
    raise NotImplementedError

labels = np.array(labels, dtype=int)
lengths = [len(d) for d in data]

print('Data for %d subjects has been loaded' % len(data))
num_subjects = len(data)
print(pd.Series(labels).value_counts())

args.modality = 'eeg_eog_emg' # eeg_eog_emg, eeg_eog, eeg, eog, emg
feature_cols = pd.read_csv('features_yasa_cols.csv', header=None).values.flatten().tolist()
print('Number of features:', len(feature_cols))
print('Feature columns:', feature_cols)

In [ ]:
import os, sys; sys.path.append(os.getcwd())
from common.stateagg import fit_states, align_states, aggregate_states

args.seed = 1
np.random.seed(args.seed)

n_clusters = 5

# Reference state centroids, estimated from the development cohort (Batch 1) only.
# The validation cohort (Batch 2) is not used for state learning. Fold-specific states
# in cross-validation are matched to these reference states (Hungarian algorithm),
# so state numbering is the same in all analyses.
cluster_centers = np.load(os.path.join(args.data_dir, 'output/kmeans_psg/session_20250711115334/cluster_centers.npz'))['arr_0']
# Session of the main_msa_pd_all.ipynb run whose Top-20 subset is transferred.
# Update after every rerun of main_msa_pd_all.ipynb (path printed by its Top-20 cell).
top20_session = 'session_20261020220701'
top20_file = os.path.join(args.data_dir, 'output', 'kmeans_psg', top20_session, 'top20_all_batch1.csv')

# State-wise features computed with the reference centroids, used to fit the
# final model on early-stage Batch 1 subjects and to evaluate it on Batch 2
features_clusters = aggregate_states(data, cluster_centers)
print(features_clusters.shape)

In [ ]:
features = features_clusters.reshape(num_subjects, -1)
print(features.shape)

feature_cols_clusters = [f"{cluster}_{col}" for cluster in range(n_clusters) for col in feature_cols]
print(len(feature_cols_clusters))
print(feature_cols_clusters)

df_features = pd.DataFrame(features, columns=feature_cols_clusters)

In [ ]:
subjects_pd_hy_0_train = pd.read_csv(os.path.join(args.data_dir, 'earlyPD_train.txt'), header=None)[0].values.tolist()
subjects_pd_hy_0_test = pd.read_csv(os.path.join(args.data_dir, 'earlyPD_test.txt'), header=None)[0].values.tolist()
subjects_msa_hy_0_train = pd.read_csv(os.path.join(args.data_dir, 'earlyMSA_train.txt'), header=None)[0].values.tolist()
subjects_msa_hy_0_test = pd.read_csv(os.path.join(args.data_dir, 'earlyMSA_test.txt'), header=None)[0].values.tolist()
print(f"Number of early PD subjects: {len(subjects_pd_hy_0_train)} : {len(subjects_pd_hy_0_test)}")
print(f"Number of early MSA subjects: {len(subjects_msa_hy_0_train)} : {len(subjects_msa_hy_0_test)}")

subjects_pd_msa_hy_0_train = subjects_pd_hy_0_train + subjects_msa_hy_0_train
idx_pd_msa_hy_0_train = [i for i, sub in enumerate(subjects) if sub in subjects_pd_msa_hy_0_train]
print(f"Number of early PD+MSA subjects in training set: {len(idx_pd_msa_hy_0_train)}")

subjects_pd_msa_hy_0_test = subjects_pd_hy_0_test + subjects_msa_hy_0_test
idx_pd_msa_hy_0_test = [i for i, sub in enumerate(subjects) if sub in subjects_pd_msa_hy_0_test]
print(f"Number of early PD+MSA subjects in test set: {len(idx_pd_msa_hy_0_test)}")

In [ ]:
# Split train and test sets
idx_train = idx_pd_msa_hy_0_train
idx_test = idx_pd_msa_hy_0_test

features_train, features_test = features[idx_train], features[idx_test]
labels_train, labels_test = labels[idx_train], labels[idx_test]
print('Train and test features shapes:')
print(features_train.shape)
print(features_test.shape)

labels_train[labels_train == args.target_labels[0]] = 0 # PD
labels_train[labels_train == args.target_labels[1]] = 1 # MSA
labels_test[labels_test == args.target_labels[0]] = 0 # PD
labels_test[labels_test == args.target_labels[1]] = 1 # MSA

In [ ]:
import csv
import shap
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix, recall_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from common import metrics


X = features_train.reshape(len(features_train), -1)
y = np.array(labels_train, dtype=int)

n_seeds = 10
shap_values_all = np.zeros((n_seeds, *X.shape), dtype=float)

data_dev = [data[k] for k in idx_train]  # same subject order as features_train
align_stats = []
centers_cv = {}  # fold-specific states, reused in the next cell

scalers = []
linkeds = []
cols_clusters_cv = []
cols_selected_cv = []
models_1, models_2 = [], []
yy_cv_trues, yy_cv_probs_1, yy_cv_probs_2 = [], [], []
scores_auc_cv_1, scores_accu_cv_1 = [], []
scores_auc_cv_2, scores_accu_cv_2 = [], []
# operating thresholds from all repeated partitions (averaged below and in the validation cell)
threshs_1, fprs_1, tprs_1 = [], [], []
threshs_2, fprs_2, tprs_2 = [], [], []

for i in range(n_seeds):

    args.seed = i + 51

    params_xgb = {
        'learning_rate': 0.02,            # 学习率，控制每一步的步长，用于防止过拟合。典型值范围：0.01 - 0.1
        'booster': 'gbtree',              # 提升方法，这里使用梯度提升树（Gradient Boosting Tree）
        'objective': 'binary:logistic',   # 损失函数，这里使用逻辑回归，用于二分类任务 'binary:logistic'
        'max_leaves': 127,                # 每棵树的叶子节点数量，控制模型复杂度。较大值可以提高模型复杂度但可能导致过拟合
        'verbosity': 1,                   # 控制 XGBoost 输出信息的详细程度，0表示无输出，1表示输出进度信息
        'seed': args.seed,                # 随机种子，用于重现模型的结果
        'nthread': -1,                    # 并行运算的线程数量，-1表示使用所有可用的CPU核心
        'colsample_bytree': 0.6,          # 每棵树随机选择的特征比例，用于增加模型的泛化能力
        'subsample': 0.7,                 # 每次迭代时随机选择的样本比例，用于增加模型的泛化能力
        'eval_metric': 'logloss'          # 评价指标，这里使用对数损失（logloss）
    }
    params_xgb['objective'] = 'binary:logistic'
    params_xgb['num_class'] = 1  # 二分类时 num_class 设置为 1
    params_xgb['eval_metric'] = 'logloss'  # 二分类时使用 logloss 作为评估指标
    
    n_folds = 5
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=args.seed)

    y_cv_trues = []
    y_cv_preds_1, y_cv_probs_1 = [], []
    y_cv_preds_2, y_cv_probs_2 = [], []
    scores_auc_1, scores_accu_1 = [], []
    scores_auc_2, scores_accu_2 = [], []

    for fold, (train_index, test_index) in enumerate(kf.split(X, y)):

        # State learning within the training fold only. Fold-specific states are matched
        # to the reference states so that each state-wise feature has one definition across folds.
        centers_fold = fit_states([data_dev[k] for k in train_index], n_clusters, random_state=args.seed * n_folds + fold)
        centers_fold, corr_fold = align_states(centers_fold, cluster_centers)
        align_stats.append([args.seed, fold] + corr_fold.tolist())
        centers_cv[(args.seed, fold)] = (centers_fold, corr_fold)
        X_fold = aggregate_states(data_dev, centers_fold).reshape(len(data_dev), -1)

        X_train, X_test = X_fold[train_index], X_fold[test_index]
        y_train, y_test = y[train_index], y[test_index]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        scalers.append(scaler)

        class_weights = len(y_train) / (len(np.unique(y_train)) * np.bincount(y_train))
        weights = np.array([class_weights[label] for label in y_train])

        # initial training with all features
        model_1 = xgb.XGBClassifier(**params_xgb)
        model_1.fit(X_train, y_train, sample_weight=weights)
        models_1.append(model_1)

        y_pred_1 = model_1.predict(X_test)
        y_prob_1 = model_1.predict_proba(X_test)[:, 1]

        y_cv_trues += y_test.tolist()
        y_cv_preds_1 += y_pred_1.tolist()
        y_cv_probs_1 += y_prob_1.tolist()
        
        auc_1 = roc_auc_score(y_test, y_prob_1)
        accu_1 = balanced_accuracy_score(y_test, y_pred_1)
        scores_auc_1.append(auc_1)
        scores_accu_1.append(accu_1)

        # compute feature importances
        explainer = shap.TreeExplainer(model_1)
        shap_values = explainer.shap_values(X_train)
        if isinstance(shap_values, list):  # 多分类情况
            shap_values = shap_values[1]  # 取正类 SHAP 值
        shap_values_all[i, train_index] = shap_values
        importances = np.abs(shap_values).mean(axis=0) if isinstance(shap_values, np.ndarray) else np.abs(shap_values).mean(axis=1)

        idx_sorted = np.abs(shap_values).mean(axis=0).argsort()[::-1]  # Sort by mean absolute SHAP values
        feature_names = df_features.columns[idx_sorted]
        feature_scores = importances[idx_sorted]

        # first select top-k features based on feature importances
        topk = 200
        idx_selected_1 = idx_sorted[:topk]
        data_selected_1 = X_train[:, idx_selected_1]
        cols_selected_1 = feature_names[:topk].tolist()
        scores_selected = feature_scores[:topk]

        # Perform hierarchical clustering on features
        data_transposed = data_selected_1.transpose()
        linked = linkage(data_transposed, method='ward', metric='euclidean')
        linkeds.append(linked)

        n_groups = 10  # Adjust the number of clusters as needed
        feature_labels = fcluster(linked, t=n_groups, criterion='maxclust')
        df_clusters = pd.DataFrame(list(zip(cols_selected_1, feature_labels, scores_selected)),
                                columns=['feature','cluster','importance'])
        df_clusters.sort_values(by=['cluster', 'importance'], ascending=[True,False], inplace=True)
        df_clusters.reset_index(drop=True, inplace=True)
        cols_clusters = [list(feats[1]) for feats in df_clusters.groupby('cluster')['feature']]
        cols_clusters = [col for cols_cluster in cols_clusters for col in cols_cluster]
        cols_clusters_cv.append(cols_clusters)
        cols_selected_2 = df_clusters.groupby('cluster').head(2)['feature'].tolist()

        # cols_selected_2 = cols_selected_1  # Use the same features for the second stage
        idx_selected_2 = [df_features.columns.get_loc(col) for col in cols_selected_2 if col in df_features.columns]
        cols_selected_cv.append(cols_selected_2)
        print(f"Selected features: {cols_selected_2}")

        model_2 = xgb.XGBClassifier(**params_xgb)
        model_2.fit(X_train[:, idx_selected_2], y_train, sample_weight=weights)
        models_2.append(model_2)

        y_pred_2 = model_2.predict(X_test[:, idx_selected_2])
        y_prob_2 = model_2.predict_proba(X_test[:, idx_selected_2])[:, 1]

        y_cv_preds_2 += y_pred_2.tolist()
        y_cv_probs_2 += y_prob_2.tolist()
        
        auc_2 = roc_auc_score(y_test, y_prob_2)
        accu_2 = balanced_accuracy_score(y_test, y_pred_2)
        scores_auc_2.append(auc_2)
        scores_accu_2.append(accu_2)

    df_yy_1 = pd.DataFrame({
        'y_true': y_cv_trues,
        'y_prob': y_cv_probs_1,
    })
    df_yy_1.to_csv(os.path.join(args.output_dir, 'yy_' + args.arch + f'_{i}_1.csv'))

    conf_mats_1 = confusion_matrix(y_cv_trues, y_cv_preds_1)
    # print(conf_mats_1)

    with open(os.path.join(args.output_dir, f'confusion_matrix_{i}_1.txt'), 'w') as f:
        csv.writer(f, delimiter=' ').writerows(conf_mats_1)
        f.close()

    auc_1 = roc_auc_score(y_cv_trues, y_cv_probs_1)
    thresh_1, fpr_1, tpr_1 = metrics.select_threshold_by_balanced_accuracy(y_cv_trues, y_cv_probs_1)
    threshs_1.append(thresh_1)
    fprs_1.append(fpr_1)
    tprs_1.append(tpr_1)

    y_cv_preds_1 = (y_cv_probs_1 >= thresh_1).astype(int)
    sensi_1 = recall_score(y_cv_trues, y_cv_preds_1, pos_label=1)
    speci_1 = recall_score(y_cv_trues, y_cv_preds_1, pos_label=0)
    accu_1 = balanced_accuracy_score(y_cv_trues, y_cv_preds_1)

    print(
        f"Seed {args.seed} result:\n"
        f"AUC mean +/- std. dev.: "
        f"{auc_1:.3f} +/- {np.std(scores_auc_1):.3f}\n"
        f"Balanced accuracy mean +/- std. dev.: "
        f"{accu_1:.3f} +/- {np.std(scores_accu_1):.3f}\n"
        f"Threshold: {thresh_1:.3f}, Sensitivity: {sensi_1:.3f}, Specificity: {speci_1:.3f}\n"
    )

    df_yy_2 = pd.DataFrame({
        'y_true': y_cv_trues,
        'y_prob': y_cv_probs_2,
    })
    df_yy_2.to_csv(os.path.join(args.output_dir, 'yy_' + args.arch + f'_{i}_2.csv'))

    conf_mats_2 = confusion_matrix(y_cv_trues, y_cv_preds_2)
    # print(conf_mats_2)

    with open(os.path.join(args.output_dir, f'confusion_matrix_{i}_2.txt'), 'w') as f:
        csv.writer(f, delimiter=' ').writerows(conf_mats_2)
        f.close()

    auc_2 = roc_auc_score(y_cv_trues, y_cv_probs_2)
    thresh_2, fpr_2, tpr_2 = metrics.select_threshold_by_balanced_accuracy(y_cv_trues, y_cv_probs_2)
    threshs_2.append(thresh_2)
    fprs_2.append(fpr_2)
    tprs_2.append(tpr_2)
    
    y_cv_preds_2 = (y_cv_probs_2 >= thresh_2).astype(int)
    sensi_2 = recall_score(y_cv_trues, y_cv_preds_2, pos_label=1)
    speci_2 = recall_score(y_cv_trues, y_cv_preds_2, pos_label=0)
    accu_2 = balanced_accuracy_score(y_cv_trues, y_cv_preds_2)

    y_cv_trues = np.array(y_cv_trues, dtype=int)
    y_cv_probs_1 = np.array(y_cv_probs_1, dtype=float)
    y_cv_probs_2 = np.array(y_cv_probs_2, dtype=float)

    yy_cv_trues += y_cv_trues.tolist()
    yy_cv_probs_1 += y_cv_probs_1.tolist()
    yy_cv_probs_2 += y_cv_probs_2.tolist()

    scores_auc_cv_1.append(auc_1)
    scores_accu_cv_1.append(accu_1)
    scores_auc_cv_2.append(auc_2)
    scores_accu_cv_2.append(accu_2)
        
    print(
        f"AUC mean +/- std. dev.: "
        f"{auc_2:.3f} +/- {np.std(scores_auc_2):.3f}\n"
        f"Balanced accuracy mean +/- std. dev.: "
        f"{accu_2:.3f} +/- {np.std(scores_accu_2):.3f}\n"
        f"Threshold: {thresh_2:.3f}, Sensitivity: {sensi_2:.3f}, Specificity: {speci_2:.3f}\n"
    )

df_align = pd.DataFrame(align_stats, columns=['seed', 'fold'] + [f'state_{j}' for j in range(n_clusters)])
df_align.to_csv(os.path.join(args.output_dir, 'state_alignment_cv.csv'), index=False)
corr_matched = df_align[[f'state_{j}' for j in range(n_clusters)]].values
print(f"Matched |r| between fold and reference states: median {np.median(corr_matched):.3f}, minimum {corr_matched.min():.3f}")

yy_cv_trues = np.array(yy_cv_trues, dtype=int)
yy_cv_probs_1 = np.array(yy_cv_probs_1, dtype=float)
yy_cv_probs_2 = np.array(yy_cv_probs_2, dtype=float)
results_cv_1 = metrics.compute_metrics(yy_cv_trues, yy_cv_probs_1, threshold=np.mean(threshs_1), with_ci=True)
results_cv_2 = metrics.compute_metrics(yy_cv_trues, yy_cv_probs_2, threshold=np.mean(threshs_2), with_ci=True)
print("Finaly averaged results with full features:")
print(
    f"AUC = {results_cv_1['auc'][0]:.3f} (95% CI: {results_cv_1['auc'][1]:.3f}-{results_cv_1['auc'][2]:.3f})\n"
    f"BAC = {results_cv_1['bac'][0]:.3f} (95% CI: {results_cv_1['bac'][1]:.3f}-{results_cv_1['bac'][2]:.3f})\n"
    f"SEN = {results_cv_1['sen'][0]:.3f} (95% CI: {results_cv_1['sen'][1]:.3f}-{results_cv_1['sen'][2]:.3f})\n"
    f"SPE = {results_cv_1['spe'][0]:.3f} (95% CI: {results_cv_1['spe'][1]:.3f}-{results_cv_1['spe'][2]:.3f})\n"
    f"PPV = {results_cv_1['ppv'][0]:.3f} (95% CI: {results_cv_1['ppv'][1]:.3f}-{results_cv_1['ppv'][2]:.3f})\n"
    f"NPV = {results_cv_1['npv'][0]:.3f} (95% CI: {results_cv_1['npv'][1]:.3f}-{results_cv_1['npv'][2]:.3f})\n"
)
print("Finaly averaged results with top-20 features:")
print(
    f"AUC = {results_cv_2['auc'][0]:.3f} (95% CI: {results_cv_2['auc'][1]:.3f}-{results_cv_2['auc'][2]:.3f})\n"
    f"BAC = {results_cv_2['bac'][0]:.3f} (95% CI: {results_cv_2['bac'][1]:.3f}-{results_cv_2['bac'][2]:.3f})\n"
    f"SEN = {results_cv_2['sen'][0]:.3f} (95% CI: {results_cv_2['sen'][1]:.3f}-{results_cv_2['sen'][2]:.3f})\n"
    f"SPE = {results_cv_2['spe'][0]:.3f} (95% CI: {results_cv_2['spe'][1]:.3f}-{results_cv_2['spe'][2]:.3f})\n"
    f"PPV = {results_cv_2['ppv'][0]:.3f} (95% CI: {results_cv_2['ppv'][1]:.3f}-{results_cv_2['ppv'][2]:.3f})\n"
    f"NPV = {results_cv_2['npv'][0]:.3f} (95% CI: {results_cv_2['npv'][1]:.3f}-{results_cv_2['npv'][2]:.3f})\n"
)

In [ ]:
## Agreement between fold-specific and reference states
# Uses the fold-specific centroids estimated in the cross-validation above (centers_cv,
# same seeds and partitions), matched to the reference states, and reports
# (i) the matched absolute centroid correlation and
# (ii) the fraction of epochs assigned to a different state than with the reference centroids.
from sklearn.model_selection import StratifiedKFold
from common.microstate import segment_predict
from common.stateagg import fit_states, align_states

n_seeds, n_folds = 10, 5
data_dev = [data[k] for k in idx_train]
y_dev = np.array(labels_train, dtype=int)
epochs_dev = [d.reshape(d.shape[0], -1) for d in data_dev]
labels_ref = [segment_predict(cluster_centers, x.T)[0] for x in epochs_dev]

align_stats = []
for (seed, fold), (centers_fold, corr_fold) in centers_cv.items():
    n_diff, n_all = 0, 0
    for x, lab_ref in zip(epochs_dev, labels_ref):
        lab_fold = segment_predict(centers_fold, x.T)[0]
        n_diff += np.sum(lab_fold != lab_ref)
        n_all += len(lab_ref)
    align_stats.append([seed, fold] + corr_fold.tolist() + [n_diff / n_all])

state_cols = [f'state_{j}' for j in range(n_clusters)]
df_align = pd.DataFrame(align_stats, columns=['seed', 'fold'] + state_cols + ['frac_epochs_reassigned'])
df_align.to_csv(os.path.join(args.output_dir, 'state_alignment_cv.csv'), index=False)

corr_matched = df_align[state_cols].values
print(f"Matched |r| between fold-specific and reference states: median {np.median(corr_matched):.4f}, minimum {corr_matched.min():.4f}")
print("Per-state minimum |r|:", np.round(corr_matched.min(axis=0), 4))
print(f"Fraction of epochs reassigned: median {df_align['frac_epochs_reassigned'].median():.4f}, "
      f"maximum {df_align['frac_epochs_reassigned'].max():.4f}")
print("Saved to", os.path.join(args.output_dir, 'state_alignment_cv.csv'))

In [ ]:
from collections import Counter

# Count selected features
all_selected = [feat for fold in cols_selected_cv for feat in fold]
feature_counter = Counter(all_selected)
feature_freq = {feat: count for feat, count in feature_counter.items()}
feature_shap = np.abs(shap_values_all).mean(axis=0).mean(axis=0) 
# sort features by frequency and then shap values
feature_freq = sorted(feature_freq.items(), key=lambda x: (x[1], feature_shap[df_features.columns.get_loc(x[0])]), reverse=True)

# print("Feature selection results:")
# for feat, freq in feature_freq:
#     print(f"{feat}: {freq:.2f}")

# plot top features by selection frequency
df_ff = pd.DataFrame(feature_freq, columns=['feature', 'freq'])
# df_plot = df_ff.copy()
top_n = min(20, len(df_ff))
df_plot = df_ff.iloc[:top_n].copy()

fig, ax = plt.subplots(figsize=(8, 3), dpi=300)
sns.barplot(data=df_plot, x='feature', y='freq', color='#7A8CA6', width=0.62, ax=ax)
ax.plot([-0.5, len(df_plot)], [25, 25], linestyle='--', linewidth=1, color='gray', alpha=0.6)
ax.set_ylim(0, 50)
ax.set_yticks(ticks=np.arange(0, 51, 10))
ax.set_xticks(ticks=range(len(df_plot)), labels=df_plot['feature'], fontsize=8, rotation=45, ha='right', rotation_mode='anchor')
ax.set_ylabel('Selection frequency')
# ax.set_xlabel('Feature')
ax.set_xlabel(f'Ranked features by selection frequency')
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

cols_selected = [feat[0] for feat in feature_freq][:20]
idx_selected = [df_features.columns.get_loc(col) for col in cols_selected if col in df_features.columns]
print("Top 20 features:", cols_selected)

In [ ]:
## 基于重复交叉验证所选特征进行独立测试
from sklearn.calibration import CalibratedClassifierCV
from common import metrics


# # Top-20 subset selected in the overall MSA vs PD analysis (main_msa_pd_all.ipynb),
# # transferred without re-selection
feature_cols_selected = pd.read_csv(top20_file)['feature'].tolist()
print('Top-20 loaded from', top20_file, '(modified', datetime.datetime.fromtimestamp(os.path.getmtime(top20_file)), ')')
# feature_cols_selected = cols_selected # Use the features selected from the previous stage
print('Transferred Top-20 features:', feature_cols_selected)

## Comment to select features based on CV results, and directly use the same features as in the first stage
idx_selected = [df_features.columns.get_loc(col) for col in feature_cols_selected if col in df_features.columns]

X_train_1, X_test_1 = features_train, features_test
X_train_2, X_test_2 = features_train[:, idx_selected], features_test[:, idx_selected]
y_train, y_test = labels_train, labels_test

scaler = StandardScaler()
X_train_1 = scaler.fit_transform(X_train_1)
X_test_1 = scaler.transform(X_test_1)

X_train_2 = scaler.fit_transform(X_train_2)
X_test_2 = scaler.transform(X_test_2)

thresholds = np.linspace(0.01, 0.8, 80)

thresh_1 = np.mean(threshs_1)
thresh_2 = np.mean(threshs_2)

n_seeds = 1
aucs_1, accus_1 = [], []
aucs_2, accus_2 = [], []
yy_test_trues, yy_test_probs_1, yy_test_probs_2 = [], [], []
for i in range(n_seeds):
    args.seed = i + 51
    params_xgb = {
        'learning_rate': 0.02,            # 学习率，控制每一步的步长，用于防止过拟合。典型值范围：0.01 - 0.1
        'booster': 'gbtree',              # 提升方法，这里使用梯度提升树（Gradient Boosting Tree）
        'objective': 'binary:logistic',   # 损失函数，这里使用逻辑回归，用于二分类任务 'binary:logistic'
        'max_leaves': 127,                # 每棵树的叶子节点数量，控制模型复杂度。较大值可以提高模型复杂度但可能导致过拟合
        'verbosity': 1,                   # 控制 XGBoost 输出信息的详细程度，0表示无输出，1表示输出进度信息
        'seed': args.seed,                # 随机种子，用于重现模型的结果
        'nthread': -1,                    # 并行运算的线程数量，-1表示使用所有可用的CPU核心
        'colsample_bytree': 0.6,          # 每棵树随机选择的特征比例，用于增加模型的泛化能力
        'subsample': 0.7,                 # 每次迭代时随机选择的样本比例，用于增加模型的泛化能力
        'eval_metric': 'logloss'          # 评价指标，这里使用对数损失（logloss）
    }

    class_weights = len(y_train) / (len(np.unique(y_train)) * np.bincount(y_train))
    weights = np.array([class_weights[label] for label in y_train])

    # with all features
    model_1 = xgb.XGBClassifier(**params_xgb)

    model_1.fit(X_train_1, y_train, sample_weight=weights)

    y_prob_1 = model_1.predict_proba(X_test_1)[:, 1]
    y_pred_1 = (y_prob_1 >= thresh_1).astype(int)
    conf_mat_1 = confusion_matrix(y_test, y_pred_1)

    auc_1 = roc_auc_score(y_test, y_prob_1)
    accu_1 = accuracy_score(y_test, y_pred_1)
    sensi_1 = recall_score(y_test, y_pred_1, pos_label=1)
    speci_1 = recall_score(y_test, y_pred_1, pos_label=0)
    aucs_1.append(auc_1)
    accus_1.append(accu_1)
    print(f"Seed {args.seed}:")
    print(f"Test AUC (full features): {auc_1:.3f}")
    print(f"Test Accuracy (full features): {accu_1:.3f}")

    # with selected features
    model_2 = xgb.XGBClassifier(**params_xgb)

    model_2.fit(X_train_2, y_train, sample_weight=weights)

    y_prob_2 = model_2.predict_proba(X_test_2)[:, 1]
    y_pred_2 = (y_prob_2 >= thresh_2).astype(int)
    conf_mat_2 = confusion_matrix(y_test, y_pred_2)

    auc_2 = roc_auc_score(y_test, y_prob_2)
    accu_2 = accuracy_score(y_test, y_pred_2)
    sensi_2 = recall_score(y_test, y_pred_2, pos_label=1)
    speci_2 = recall_score(y_test, y_pred_2, pos_label=0)
    aucs_2.append(auc_2)
    accus_2.append(accu_2)
    print(f"Test AUC (Top-20 features): {auc_2:.3f}")
    print(f"Test Accuracy (Top-20 features): {accu_2:.3f}")

    y_test = np.array(y_test, dtype=int)
    y_prob_1 = np.array(y_prob_1, dtype=float)
    y_prob_2 = np.array(y_prob_2, dtype=float)
    
    yy_test_trues += y_test.tolist()
    yy_test_probs_1 += y_prob_1.tolist()
    yy_test_probs_2 += y_prob_2.tolist()

yy_test_trues = np.array(yy_test_trues, dtype=int)
yy_test_probs_1 = np.array(yy_test_probs_1, dtype=float)
yy_test_probs_2 = np.array(yy_test_probs_2, dtype=float)
results_test_1 = metrics.compute_metrics(yy_test_trues, yy_test_probs_1, threshold=thresh_1, with_ci=True)
results_test_2 = metrics.compute_metrics(yy_test_trues, yy_test_probs_2, threshold=thresh_2, with_ci=True)
print("Finaly averaged results with full features:")
print(
    f"AUC = {results_test_1['auc'][0]:.3f} (95% CI: {results_test_1['auc'][1]:.3f}-{results_test_1['auc'][2]:.3f})\n"
    f"BAC = {results_test_1['bac'][0]:.3f} (95% CI: {results_test_1['bac'][1]:.3f}-{results_test_1['bac'][2]:.3f})\n"
    f"SEN = {results_test_1['sen'][0]:.3f} (95% CI: {results_test_1['sen'][1]:.3f}-{results_test_1['sen'][2]:.3f})\n"
    f"SPE = {results_test_1['spe'][0]:.3f} (95% CI: {results_test_1['spe'][1]:.3f}-{results_test_1['spe'][2]:.3f})\n"
    f"PPV = {results_test_1['ppv'][0]:.3f} (95% CI: {results_test_1['ppv'][1]:.3f}-{results_test_1['ppv'][2]:.3f})\n"
    f"NPV = {results_test_1['npv'][0]:.3f} (95% CI: {results_test_1['npv'][1]:.3f}-{results_test_1['npv'][2]:.3f})\n"
)
print("Finaly averaged results with top-20 features:")
print(
    f"AUC = {results_test_2['auc'][0]:.3f} (95% CI: {results_test_2['auc'][1]:.3f}-{results_test_2['auc'][2]:.3f})\n"
    f"BAC = {results_test_2['bac'][0]:.3f} (95% CI: {results_test_2['bac'][1]:.3f}-{results_test_2['bac'][2]:.3f})\n"
    f"SEN = {results_test_2['sen'][0]:.3f} (95% CI: {results_test_2['sen'][1]:.3f}-{results_test_2['sen'][2]:.3f})\n"
    f"SPE = {results_test_2['spe'][0]:.3f} (95% CI: {results_test_2['spe'][1]:.3f}-{results_test_2['spe'][2]:.3f})\n"
    f"PPV = {results_test_2['ppv'][0]:.3f} (95% CI: {results_test_2['ppv'][1]:.3f}-{results_test_2['ppv'][2]:.3f})\n"
    f"NPV = {results_test_2['npv'][0]:.3f} (95% CI: {results_test_2['npv'][1]:.3f}-{results_test_2['npv'][2]:.3f})\n"
)

In [ ]:
## Save validation-cohort predictions (calibration analysis)
# Predicted MSA probabilities of the final models in the temporal validation cohort,
# together with the development-derived decision thresholds.
df_yy_test = pd.DataFrame({
    'subject': [subjects[k] for k in idx_test],
    'y_true': yy_test_trues,
    'y_prob_full': yy_test_probs_1,
    'y_prob_top20': yy_test_probs_2,
})
df_yy_test['thresh_full'] = thresh_1
df_yy_test['thresh_top20'] = thresh_2
df_yy_test.to_csv(os.path.join(args.output_dir, 'yy_test.csv'), index=False)
print('Validation AUC (full / Top-20):', roc_auc_score(yy_test_trues, yy_test_probs_1), roc_auc_score(yy_test_trues, yy_test_probs_2))
print('Saved to', args.output_dir)

In [ ]:
import seaborn as sn
from sklearn.metrics import roc_curve

conditions = ['Full feature set', 'Top-20 features']
roc_curve_1 = roc_curve(y_cv_trues, y_cv_probs_1)
roc_curve_2 = roc_curve(y_cv_trues, y_cv_probs_2)
roc_curve_t_1 = roc_curve(y_test, y_prob_1)
roc_curve_t_2 = roc_curve(y_test, y_prob_2)
roc_curves = [roc_curve_1, roc_curve_2]
roc_curves_t = [roc_curve_t_1, roc_curve_t_2]
conf_mats = [conf_mats_1, conf_mats_2]
conf_mats_t = [conf_mat_1, conf_mat_2]
aucs = [results_cv_1['auc'][0], results_cv_2['auc'][0]]
accus = [results_cv_1['bac'][0], results_cv_2['bac'][0]]
aucs_t = [auc_1, auc_2]
accus_t = [accu_1, accu_2]
fprs = [np.mean(fprs_1), np.mean(fprs_2)]
tprs = [np.mean(tprs_1), np.mean(tprs_2)]

fig, axes = plt.subplots(2, 3, figsize=(8, 5.2), dpi=300)
for i in range(2):
    axes[i, 0].plot(roc_curves[i][0], roc_curves[i][1], linestyle='--', label=f"CV AUC={aucs[i]:.3f}")
    axes[i, 0].plot(roc_curves_t[i][0], roc_curves_t[i][1], color='#C44E52', label=f"Test AUC={aucs_t[i]:.3f}")
    # axes[i, 0].plot(roc_curves[i][0], roc_curves[i][1], label=f"CV AUC={aucs[i]:.3f}, BACC={accus[i]:.3f}")
    # axes[i, 0].plot(roc_curves_t[i][0], roc_curves_t[i][1], linestyle='--', label=f"Test AUC={aucs_t[i]:.3f}, BACC={accus_t[i]:.3f}")
    axes[i, 0].plot([0, 1], [0, 1], linestyle='--', color='lightgray')
    axes[i, 0].scatter(fprs[i], tprs[i], marker='x', color="#e22f0f", s=30, linewidths=2, zorder=5, alpha=0.95)
    axes[i, 0].annotate(conditions[i], xy=(-0.35, 0.5), xycoords='axes fraction', rotation=90, fontsize=12, fontweight='bold', ha='center', va='center')
    axes[i, 0].set_xlabel(r"1 - Specificity")
    axes[i, 0].set_ylabel(r"Sensitivity")
    # axes[i, 0].set_title(f'BACC={baccus[i]:.3f}, AUC={baucs[i]:.3f}')
    axes[i, 0].legend(loc='lower right')
    sn.heatmap(conf_mats[i], annot=True, cbar=False, cmap="Blues", fmt='g', ax=axes[i, 1])
    if i == 0: axes[i, 1].set_title('CV')
    axes[i, 1].set_xlabel(r"Predicted")
    axes[i, 1].set_ylabel(r"Truth")
    axes[i, 1].set_xticklabels(['PD', 'MSA'])
    axes[i, 1].set_yticklabels(['PD', 'MSA'])
    sn.heatmap(conf_mats_t[i], annot=True, cbar=False, cmap="Blues", fmt='g', ax=axes[i, 2])
    if i == 0: axes[i, 2].set_title('Test')
    axes[i, 2].set_xlabel(r"Predicted")
    axes[i, 2].set_ylabel(r"Truth")
    axes[i, 2].set_xticklabels(['PD', 'MSA'])
    axes[i, 2].set_yticklabels(['PD', 'MSA'])
plt.tight_layout()
plt.show()

# fig, axes = plt.subplots(2, 2, figsize=(5.5, 5.0), dpi=300)
# for i in range(2):
#     # axes[0, i].plot(roc_curves[i][0], roc_curves[i][1], label=f"CV AUC={aucs[i]:.3f}")
#     axes[0, i].plot(roc_curves_t[i][0], roc_curves_t[i][1], linestyle='--', color='darkorange', label=f"Test AUC={aucs_t[i]:.3f}")
#     # axes[0, i].plot(roc_curves[i][0], roc_curves[i][1], label=f"CV AUC={aucs[i]:.3f}, BACC={accus[i]:.3f}")
#     # axes[0, i].plot(roc_curves_t[i][0], roc_curves_t[i][1], linestyle='--', label=f"Test AUC={aucs_t[i]:.3f}, BACC={accus_t[i]:.3f}")
#     axes[0, i].plot([0, 1], [0, 1], linestyle='--', color='lightgray')
#     axes[0, i].scatter(fprs[i], tprs[i], marker='x', color="#e22f0f", s=30, linewidths=2, zorder=5, alpha=0.95)
#     axes[0, i].set_title(conditions[i])
#     axes[0, i].set_xlabel(r"1 - Specificity")
#     axes[0, i].set_ylabel(r"Sensitivity")
#     # axes[0, i].set_title(f'BACC={baccus[i]:.3f}, AUC={baucs[i]:.3f}')
#     axes[0, i].legend(loc='lower right')
#     axes[0, i].grid(False)
#     # sn.heatmap(conf_mats[i], annot=True, cbar=False, cmap="Blues", fmt='g', ax=axes[1, i])
#     # axes[1, i].set_xlabel(r"Predicted")
#     # axes[1, i].set_ylabel(r"Truth")
#     # axes[1, i].set_xticklabels(['PD', 'MSA'])
#     # axes[1, i].set_yticklabels(['PD', 'MSA'])
#     sn.heatmap(conf_mats_t[i], annot=True, cbar=False, cmap="Blues", fmt='g', ax=axes[1, i])
#     axes[1, i].set_xlabel(r"Predicted")
#     axes[1, i].set_ylabel(r"Truth")
#     axes[1, i].set_xticklabels(['PD', 'MSA'])
#     axes[1, i].set_yticklabels(['PD', 'MSA'])
# plt.tight_layout()
# plt.show()

In [ ]:
## Incremental value over routinely available PSG-time variables
# Reference model: age, sex, AHI and the 46 sleep macro features.
# Incremental model: reference + dynamic Top-20 features.
# Same partitions and seeds as the cross-validation above; in each training fold the dynamic
# features use the fold-specific states (centers_cv) and fold-specific selected features
# (cols_selected_cv). Missing values are imputed with training-fold medians.
from common.incremental import load_clinical, macro_features, run_incremental, REFERENCE_CLINICAL

clinical_file = os.path.join(args.data_dir, 'MSA_PD_UMSARS_moreinfo.xlsx')  # not in the repository (identifiable data)
subj_dev = [subjects[k] for k in idx_train]
subj_val = [subjects[k] for k in idx_test]
clin_dev = load_clinical(clinical_file, subj_dev)[REFERENCE_CLINICAL].values
clin_val = load_clinical(clinical_file, subj_val)[REFERENCE_CLINICAL].values
print('Missing clinical values (dev):', np.round(np.isnan(clin_dev).mean(axis=0), 2))

# Sleep macro features from the technician-scored hypnograms (0=W, 1=N1, 2=N2, 3=N3, 4=REM)
macro_dev = macro_features([labels_stage[k] for k in idx_train], subj_dev).values
macro_val = macro_features([labels_stage[k] for k in idx_test], subj_val).values
X_ref_dev = np.hstack([clin_dev, macro_dev])
X_ref_val = np.hstack([clin_val, macro_val])

cv_keys = list(centers_cv.keys())  # (seed, fold), same order as cols_selected_cv
def dyn_cv(k, train_index):
    centers_fold = centers_cv[cv_keys[k]][0]
    X_fold = aggregate_states(data_dev, centers_fold).reshape(len(data_dev), -1)
    idx = [df_features.columns.get_loc(c) for c in cols_selected_cv[k]]
    return X_fold[:, idx]

res_inc, diff_inc = run_incremental(
    X_ref_dev, labels_train, cv_seeds=range(51, 61), dyn_cv=dyn_cv,
    X_dyn_dev=features_train[:, idx_selected],
    test=dict(ref=X_ref_val, dyn=features_test[:, idx_selected], y=labels_test),
    final_seed=20)
pd.set_option('display.precision', 3)
print(res_inc.to_string())
print(diff_inc.to_string())
res_inc.to_csv(os.path.join(args.output_dir, 'incremental_auc.csv'))
diff_inc.to_csv(os.path.join(args.output_dir, 'incremental_diff.csv'))
print('Saved to', args.output_dir)
